In [ ]:
import pandas as pd
df = pd.read_csv('../data/Chinese_Restaurants_20260720.csv')
df.head()


In [ ]:
eateries = pd.read_json('../data/DPR_Eateries_001.json')
print(eateries.shape)
print(eateries.columns.tolist())
eateries.head()


In [ ]:
# ---- Build "worst grade per restaurant" table from the inspection CSV ----
insp = pd.read_csv('../data/Chinese_Restaurants_20260720.csv')
insp['INSPECTION DATE'] = pd.to_datetime(insp['INSPECTION DATE'], errors='coerce')

# NYC letter grades: A (best) -> B -> C (worst). N/Z/P and blanks aren't final grades, so ignore them.
grade_rank = {'A': 1, 'B': 2, 'C': 3}
insp['grade_rank'] = insp['GRADE'].map(grade_rank)

# One row per restaurant (CAMIS = unique restaurant id), summarizing its whole inspection history.
agg = insp.groupby('CAMIS').agg(
    DBA=('DBA', 'first'),
    BORO=('BORO', 'first'),
    STREET=('STREET', 'first'),
    ZIPCODE=('ZIPCODE', 'first'),
    worst_score=('SCORE', 'max'),                 # higher score = more violations = worse
    first_inspection=('INSPECTION DATE', 'min'),
    last_inspection=('INSPECTION DATE', 'max'),
    n_inspections=('INSPECTION DATE', 'count'),
)

# Worst letter grade ever received (max rank among A/B/C rows only).
rank_to_grade = {1: 'A', 2: 'B', 3: 'C'}
worst_grade = (insp.dropna(subset=['grade_rank'])
                   .groupby('CAMIS')['grade_rank'].max()
                   .map(rank_to_grade)
                   .rename('worst_grade'))

worst = agg.join(worst_grade).reset_index()
worst.to_csv('../data/chinese_worst_grades.csv', index=False)
print('restaurants:', len(worst), '| with a letter grade:', worst['worst_grade'].notna().sum())
worst.head()


In [ ]:
# ---- Clean inspection dataset: keep only A/B/C rows, worst grade on top ----
graded = pd.read_csv('../data/Chinese_Restaurants_20260720.csv')
graded['INSPECTION DATE'] = pd.to_datetime(graded['INSPECTION DATE'], errors='coerce')

# Keep only real letter grades.
graded = graded[graded['GRADE'].isin(['A', 'B', 'C'])].copy()

# Sort worst-first: C -> B -> A, and within a grade the higher (worse) score first.
graded['grade_rank'] = graded['GRADE'].map({'A': 1, 'B': 2, 'C': 3})
graded = graded.sort_values(['grade_rank', 'SCORE'], ascending=[False, False]).drop(columns='grade_rank')

graded.to_csv('../data/chinese_restaurants_graded.csv', index=False)
print('rows:', len(graded), '| grade counts:')
print(graded['GRADE'].value_counts())
graded.head()


In [ ]:
# ---- Deduplicated: one row per INSPECTION (CAMIS + inspection date) ----
# The graded file has one row per violation; collapse each inspection to a single row,
# summarizing the violations as counts so no info is lost.
graded['is_critical'] = graded['CRITICAL FLAG'].eq('Critical')

inspections = (graded
    .groupby(['CAMIS', 'INSPECTION DATE'], as_index=False)
    .agg(DBA=('DBA', 'first'),
         BORO=('BORO', 'first'),
         STREET=('STREET', 'first'),
         ZIPCODE=('ZIPCODE', 'first'),
         CUISINE=('CUISINE DESCRIPTION', 'first'),
         SCORE=('SCORE', 'first'),        # score is the inspection total, same on every row
         GRADE=('GRADE', 'first'),
         INSPECTION_TYPE=('INSPECTION TYPE', 'first'),
         n_violations=('VIOLATION CODE', 'count'),
         n_critical=('is_critical', 'sum')))

# Worst inspections on top: C -> B -> A, then higher score first.
inspections['grade_rank'] = inspections['GRADE'].map({'A': 1, 'B': 2, 'C': 3})
inspections = inspections.sort_values(['grade_rank', 'SCORE'], ascending=[False, False]).drop(columns='grade_rank')

inspections.to_csv('../data/chinese_inspections_dedup.csv', index=False)
print('inspections:', len(inspections), '| grade counts:')
print(inspections['GRADE'].value_counts())
inspections.head()


In [ ]:
# ---- Merge worst-grade table with the review sheet (join on restaurant name) ----
rev = pd.read_excel('../data/chinese-restaurant-review-nyc.xlsx')

# The two files share no ID, so we join on a normalized name key.
def name_key(s):
    return (s.astype(str).str.upper().str.strip()
             .str.replace(r'[^A-Z0-9 ]', '', regex=True)   # drop punctuation
             .str.replace(r'\s+', ' ', regex=True))         # collapse spaces

worst['name_key'] = name_key(worst['DBA'])
rev['name_key'] = name_key(rev['Restaurant'])

merged = worst.merge(rev, on='name_key', how='inner')
merged = merged.drop_duplicates('CAMIS')   # guard against duplicate name matches

print(f'matched {merged["CAMIS"].nunique()} of {len(worst)} restaurants '
      f'({merged["CAMIS"].nunique() / len(worst):.0%})')
merged.to_csv('../data/worst_grade_vs_review.csv', index=False)
merged[['DBA', 'worst_grade', 'worst_score',
        'Chinese Reviews Rating', 'All Reviews Rating', 'All Reviews Count']].head(15)


In [ ]:
# ---- Build an assisted worksheet for MANUAL matching of the leftovers ----
import difflib

# Restaurants that did NOT auto-match, and review rows not yet used.
matched_keys = set(merged['name_key'])
unmatched_worst = worst[~worst['name_key'].isin(matched_keys)].copy()
unmatched_rev = rev[~rev['name_key'].isin(matched_keys)].copy()

# Candidate pool: map each unmatched review name_key back to its original display name.
rev_names = unmatched_rev.drop_duplicates('name_key').set_index('name_key')['Restaurant']
rev_keys = rev_names.index.tolist()

def suggest(key, n=3):
    hits = difflib.get_close_matches(key, rev_keys, n=n, cutoff=0.6)  # 0.0-1.0 similarity
    return [rev_names[h] for h in hits]

sugg = unmatched_worst['name_key'].apply(suggest)
for i in range(3):
    unmatched_worst[f'suggestion_{i+1}'] = sugg.apply(lambda s: s[i] if len(s) > i else '')

# Rows with a suggestion first (easiest to confirm), then the rest.
unmatched_worst['has_suggestion'] = unmatched_worst['suggestion_1'] != ''
sheet = unmatched_worst.sort_values(['has_suggestion', 'DBA'], ascending=[False, True])

# 'matched_review' is the column YOU fill in: paste the correct review name (or leave blank).
sheet['matched_review'] = ''
cols = ['CAMIS', 'DBA', 'BORO', 'STREET', 'ZIPCODE', 'worst_grade', 'worst_score',
        'suggestion_1', 'suggestion_2', 'suggestion_3', 'matched_review']
sheet[cols].to_csv('../data/match_worksheet.csv', index=False)

print(f'unmatched restaurants: {len(unmatched_worst)} | '
      f'{sheet["has_suggestion"].sum()} have a suggested candidate')
print('-> edit ../data/match_worksheet.csv, fill the "matched_review" column, then run the next cell')
sheet[cols].head(15)


In [ ]:
# ---- Apply your manual matches and combine with the auto-matches ----
# Run this AFTER filling the 'matched_review' column in ../data/match_worksheet.csv.
filled = pd.read_csv('../data/match_worksheet.csv')
filled = filled[filled['matched_review'].notna() & (filled['matched_review'].astype(str).str.strip() != '')]

# Attach each hand-picked review name to its inspection restaurant (by CAMIS), then join the review data.
manual = (filled[['CAMIS', 'matched_review']]
          .assign(name_key=lambda d: name_key(d['matched_review']))
          .merge(worst.drop(columns='name_key'), on='CAMIS', how='left')
          .merge(rev, on='name_key', how='left'))

# Stack auto-matches (from the earlier merge) + manual matches into one table.
final = (pd.concat([merged, manual], ignore_index=True)
           .drop_duplicates('CAMIS'))
final.to_csv('../data/worst_grade_vs_review_final.csv', index=False)

print(f'auto: {merged["CAMIS"].nunique()} | manual: {len(filled)} | '
      f'total: {final["CAMIS"].nunique()} of {len(worst)} ({final["CAMIS"].nunique()/len(worst):.0%})')
final[['DBA', 'worst_grade', 'worst_score',
       'Chinese Reviews Rating', 'All Reviews Rating', 'All Reviews Count']].head(15)
